# Création de paquets

Ce TP fait d'un petit module Python un paquet distribuable, avec `uv` : un `pyproject.toml` qui déclare son back-end de construction et une commande, une installation éditable pour le développement, puis la construction de l'archive source et de la roue, leur installation dans un environnement neuf et la livraison d'une nouvelle version. Le paquet, `thermometre`, convertit des températures de degrés Celsius en degrés Fahrenheit avec la commande `vers-fahrenheit`.

Le TP tourne dans Colab comme dans le Codespace. Une ligne qui commence par `!` est une commande du shell, et une cellule qui commence par `%%writefile <fichier>` écrit son contenu dans ce fichier. Les étapes s'enchaînent : chacune part du paquet de la précédente. La dernière, la publication sur Test PyPI, est facultative et ne s'exécute pas avec « Exécuter tout ».

In [ ]:
# @title À exécuter : mise en place de l'environnement
!rm -rf sample_data
!pip install -q -U uv

## 1. Créer le squelette du paquet

Créez avec `uv init` le squelette d'une bibliothèque nommée `thermometre`, avec le back-end de construction par défaut d'uv, celui des slides, puis placez-vous dans son dossier avec la commande magique `%cd`. Affichez ensuite la liste de ses fichiers (hors `.git`) et son `pyproject.toml`.

Où uv place-t-il le code du paquet ? Quelle table du `pyproject.toml` désigne le back-end de construction, et que contient-elle ?

**Documentation utile**

- https://docs.astral.sh/uv/reference/cli/#uv-init
- https://docs.astral.sh/uv/concepts/projects/init/#libraries
- https://docs.astral.sh/uv/concepts/build-backend/

In [ ]:
# Votre code ici

### Conseils supplémentaires

`uv init --lib` crée une bibliothèque : le code dans `src/thermometre/` et une table `[build-system]`, sans commande. Sans option, uv crée une application installable, qui déclare déjà une commande, et `--no-package` un projet qu'on n'installe pas, sans table `[build-system]`. Le back-end par défaut est celui d'uv, `uv_build` (`--build-backend` en choisit un autre, comme `hatchling`). `find . -type f -not -path "./.git/*"` liste les fichiers sans ceux du dépôt git que crée uv.

```python
!uv init ... thermometre
%cd thermometre
!find . -type f -not -path "./.git/*"
!cat ...
```

### Solution

In [ ]:
!uv init --lib thermometre

In [ ]:
%cd thermometre

In [ ]:
!find . -type f -not -path "./.git/*"
!cat pyproject.toml

Le code est dans `src/thermometre/` (le nom du projet, avec des `_` à la place des `-`) : `__init__.py` en fait un paquet, et `py.typed` annonce aux vérificateurs de types qu'il est annoté. La table `[build-system]` nomme le back-end (`build-backend = "uv_build"`) et ce qu'il faut installer pour le lancer (`requires = ["uv_build>=…,<…"]`, une plage de versions) : `uv`, `pip` ou tout autre front-end le lisent pour construire le paquet, sans rien savoir d'`uv_build`.

## 2. Écrire le code et déclarer la commande

Écrivez dans le paquet deux modules :

- `src/thermometre/conversion.py`, avec une fonction `celsius_vers_fahrenheit(celsius: float) -> float` (F = C × 9 / 5 + 32) ;
- `src/thermometre/cli.py`, avec une fonction `main()` qui convertit chacun des arguments de la ligne de commande et affiche une ligne comme `20.0 °C = 68.0 °F`.

Réécrivez ensuite `pyproject.toml` en y ajoutant une table `[project.scripts]` qui déclare la commande `vers-fahrenheit`, appelant `main`. Gardez les tables `[project]` et `[build-system]` qu'y a mises uv (la description, les auteurs et le reste à votre goût).

**Documentation utile**

- https://docs.python.org/fr/3/library/sys.html#sys.argv
- https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#creating-executable-scripts
- https://docs.astral.sh/uv/concepts/projects/config/#command-line-interfaces

In [ ]:
# Votre code ici

### Conseils supplémentaires

`sys.argv[1:]` donne les arguments de la commande, des chaînes à convertir avec `float`. Une entrée de `[project.scripts]` associe le nom d'une commande à `"<module>:<fonction>"`, ici `"thermometre.cli:main"`. Réécrire tout le fichier avec `%%writefile`, plutôt que d'y ajouter la table, permet de relancer la cellule sans la dupliquer. Gardez la table `[build-system]`.

```python
%%writefile src/thermometre/cli.py
import sys

from thermometre.conversion import celsius_vers_fahrenheit


def main() -> None:
  for argument in ...:
    celsius = ...
    print(f"{celsius} °C = {...} °F")
```

```toml
[project.scripts]
vers-fahrenheit = "..."
```

### Solution

In [ ]:
%%writefile src/thermometre/conversion.py
def celsius_vers_fahrenheit(celsius: float) -> float:
  return celsius * 9 / 5 + 32

In [ ]:
%%writefile src/thermometre/cli.py
import sys

from thermometre.conversion import celsius_vers_fahrenheit


def main() -> None:
  for argument in sys.argv[1:]:
    celsius = float(argument)
    print(f"{celsius} °C = {celsius_vers_fahrenheit(celsius)} °F")

In [ ]:
%%writefile pyproject.toml
[project]
name = "thermometre"
version = "0.1.0"
description = "Conversions de températures."
readme = "README.md"
requires-python = ">=3.12"
dependencies = []

[project.scripts]
vers-fahrenheit = "thermometre.cli:main"

[build-system]
requires = ["uv_build>=0.13.1,<0.14.0"]
build-backend = "uv_build"

## 3. Installer le paquet en mode éditable

Installez le paquet en mode éditable avec `uv pip install`, puis lancez `vers-fahrenheit 20 36.6 -40`. Que vaut la conversion de 36,6 °C ?

Corrigez l'affichage dans `cli.py`, avec une décimale pour les deux températures, et relancez la commande sans réinstaller le paquet. Pourquoi la correction s'applique-t-elle ?

**Documentation utile**

- https://docs.astral.sh/uv/reference/cli/#uv-pip-install--editable

In [ ]:
# Votre code ici

### Conseils supplémentaires

`-e .` installe en mode éditable le paquet du dossier courant. Dans Colab et le Codespace, `uv pip install` installe dans le Python du système, celui du notebook, dont le dossier des commandes est dans le `PATH`. Le calcul de 36,6 × 9 / 5 + 32 en flottants donne `97.88000000000001` ; la spécification `:.1f` d'une f-string affiche une décimale.

```python
!uv pip install ...
!vers-fahrenheit 20 36.6 -40
```

### Solution

In [ ]:
!uv pip install -e .
!vers-fahrenheit 20 36.6 -40

In [ ]:
%%writefile src/thermometre/cli.py
import sys

from thermometre.conversion import celsius_vers_fahrenheit


def main() -> None:
  for argument in sys.argv[1:]:
    celsius = float(argument)
    print(f"{celsius:.1f} °C = {celsius_vers_fahrenheit(celsius):.1f} °F")

In [ ]:
!vers-fahrenheit 20 36.6 -40

36,6 °C donne `97.88000000000001 °F` : 36,6 n'a pas d'écriture exacte en binaire. Une installation éditable ne copie pas le code : elle laisse dans l'environnement un chemin vers `src/`, et Python lit donc les sources à chaque import. La commande `vers-fahrenheit`, un petit script qu'a créé l'installation, importe `thermometre.cli` et appelle `main` : elle voit chaque modification du code sans réinstallation (pas celles du `pyproject.toml`, comme une nouvelle commande, qui demandent de réinstaller).

## 4. Construire les distributions

Construisez avec `uv build` l'archive source (*sdist*, `.tar.gz`) et la roue (*wheel*, `.whl`) du paquet, dans `dist/`. Listez le contenu de chacune avec les interfaces en ligne de commande des modules `tarfile` et `zipfile`, puis extrayez la roue dans `../roue` et affichez ses fichiers `METADATA` et `entry_points.txt`.

Quels fichiers sont dans l'archive et pas dans la roue, et inversement ? Où la roue garde-t-elle la version du paquet et sa commande ?

**Documentation utile**

- https://docs.astral.sh/uv/reference/cli/#uv-build
- https://docs.python.org/fr/3/library/tarfile.html#command-line-interface
- https://docs.python.org/fr/3/library/zipfile.html#command-line-interface

In [ ]:
# Votre code ici

### Conseils supplémentaires

`uv build` construit les deux par défaut, la roue à partir de l'archive. `python -m tarfile -l <archive>` et `python -m zipfile -l <roue>` listent le contenu, `python -m zipfile -e <roue> <dossier>` extrait la roue (une roue est une archive zip). Les fichiers de `dist/` s'appellent `thermometre-0.1.0.tar.gz` et `thermometre-0.1.0-py3-none-any.whl`, et les métadonnées de la roue sont dans son dossier `thermometre-0.1.0.dist-info/`.

```python
!uv build
!python -m tarfile -l dist/...
!python -m zipfile -l dist/...
!python -m zipfile -e dist/... ../roue
!cat ../roue/.../METADATA ../roue/.../entry_points.txt
```

### Solution

In [ ]:
!uv build
!python -m tarfile -l dist/thermometre-0.1.0.tar.gz
!python -m zipfile -l dist/thermometre-0.1.0-py3-none-any.whl

In [ ]:
!python -m zipfile -e dist/thermometre-0.1.0-py3-none-any.whl ../roue
!cat ../roue/thermometre-0.1.0.dist-info/METADATA
!cat ../roue/thermometre-0.1.0.dist-info/entry_points.txt

L'archive source contient les sources telles quelles, `pyproject.toml` compris, plus un `PKG-INFO` : l'installer demande de construire le paquet, donc de lancer son back-end. La roue est prête à installer : il suffit de copier ses fichiers à leur place. Elle contient le code du paquet, sans `src/` ni `pyproject.toml`, et un dossier `.dist-info` : `METADATA` reprend la table `[project]` (nom, version, version de Python requise, et les dépendances dans des lignes `Requires-Dist`), `entry_points.txt` la commande de `[project.scripts]`, et `RECORD` la liste des fichiers. Le nom de la roue dit pour quels Python et quelles plates-formes elle vaut : `py3-none-any`, du Python pur pour toutes. PyPI reçoit les deux : la roue pour installer vite, l'archive pour les plates-formes qu'aucune roue ne couvre.

## 5. Installer la roue dans un environnement neuf

Créez avec `uv venv` un environnement virtuel neuf dans `../essai-roue`, hors du dossier du paquet, installez-y la roue, puis lancez la commande `vers-fahrenheit` de cet environnement sur `100`.

**Documentation utile**

- https://docs.astral.sh/uv/reference/cli/#uv-venv
- https://docs.astral.sh/uv/reference/cli/#uv-pip-install

In [ ]:
# Votre code ici

### Conseils supplémentaires

`--clear` vide l'environnement s'il existe déjà, au lieu de demander quoi faire. `uv pip install --python ../essai-roue/bin/python <roue>` installe dans cet environnement plutôt que dans celui du notebook, et ses commandes sont dans `../essai-roue/bin/`. La commande doit afficher `100.0 °C = 212.0 °F`.

```python
!uv venv ... ../essai-roue
!uv pip install --python ... dist/thermometre-0.1.0-py3-none-any.whl
!../essai-roue/bin/vers-fahrenheit 100
```

### Solution

In [ ]:
!uv venv --clear ../essai-roue
!uv pip install --python ../essai-roue/bin/python dist/thermometre-0.1.0-py3-none-any.whl
!../essai-roue/bin/vers-fahrenheit 100

C'est ce que fait l'utilisateur d'un paquet publié, à ceci près que `uv` ou `pip` téléchargent la roue depuis PyPI. Ici, l'environnement reçoit une copie du code de la roue : il ne dépend plus du dossier `src/`. `uvx --from dist/thermometre-0.1.0-py3-none-any.whl vers-fahrenheit 100` ferait la même chose dans un environnement jetable.

## 6. Livrer une nouvelle version

Sans argument, `vers-fahrenheit` n'affiche rien et se termine avec le code de sortie 0, celui d'un succès, comme le montre la cellule suivante.

Faites que, sans argument, `main` affiche un message d'usage sur la sortie d'erreur et renvoie 1, et 0 sinon. Comparez alors la commande de l'installation éditable et celle de `../essai-roue`. Passez ensuite le paquet à la version 0.2.0 avec `uv version`, reconstruisez-le, installez la nouvelle roue dans `../essai-roue` et vérifiez sa commande.

**Documentation utile**

- https://docs.astral.sh/uv/reference/cli/#uv-version
- https://docs.astral.sh/uv/guides/package/#updating-your-version
- https://docs.python.org/fr/3/library/sys.html#sys.stderr

In [ ]:
!../essai-roue/bin/vers-fahrenheit; echo "Code de sortie : $?"

In [ ]:
# Votre code ici

### Conseils supplémentaires

Le script d'une commande appelle `sys.exit(main())` : la valeur que renvoie `main` devient le code de sortie. `print(..., file=sys.stderr)` écrit sur la sortie d'erreur. `uv version --bump minor` passe de 0.1.0 à 0.2.0 dans le `pyproject.toml`, et `--no-sync` lui évite de créer un environnement `.venv` pour le projet. La commande éditable doit répondre 1 dès la modification, celle de `../essai-roue` 0 jusqu'à l'installation de la roue 0.2.0.

```python
%%writefile src/thermometre/cli.py
import sys

from thermometre.conversion import celsius_vers_fahrenheit


def main() -> int:
  arguments = sys.argv[1:]
  if not arguments:
    print(..., file=sys.stderr)
    return ...
  for argument in arguments:
    celsius = float(argument)
    print(f"{celsius:.1f} °C = {celsius_vers_fahrenheit(celsius):.1f} °F")
  return ...
```

```python
!vers-fahrenheit; echo "Code de sortie : $?"
!uv version ...
!uv build
!uv pip install --python ../essai-roue/bin/python dist/...
```

### Solution

In [ ]:
%%writefile src/thermometre/cli.py
import sys

from thermometre.conversion import celsius_vers_fahrenheit


def main() -> int:
  arguments = sys.argv[1:]
  if not arguments:
    print("Usage : vers-fahrenheit TEMPÉRATURE...", file=sys.stderr)
    return 1
  for argument in arguments:
    celsius = float(argument)
    print(f"{celsius:.1f} °C = {celsius_vers_fahrenheit(celsius):.1f} °F")
  return 0

In [ ]:
!vers-fahrenheit; echo "Code de sortie : $?"
!../essai-roue/bin/vers-fahrenheit; echo "Code de sortie : $?"

In [ ]:
!uv version --bump minor --no-sync
!uv build
!uv pip install --python ../essai-roue/bin/python dist/thermometre-0.2.0-py3-none-any.whl
!../essai-roue/bin/vers-fahrenheit; echo "Code de sortie : $?"

L'installation éditable suit les sources ; l'environnement `../essai-roue` garde la copie de la roue 0.1.0 jusqu'à ce qu'on y installe la 0.2.0, que `uv pip install` met à la place de l'ancienne. Une nouvelle version publiée a toujours un nouveau numéro : PyPI refuse de remplacer les fichiers d'une version. `dist/` garde les fichiers des deux versions, et `uv publish` envoie tout ce qu'il contient : videz-le avant de construire une version à publier.

## 7. Pour aller plus loin : publier sur Test PyPI

Cette étape est facultative : elle demande un compte sur [Test PyPI](https://test.pypi.org/account/register/), l'index de test de Python, distinct de PyPI, avec une adresse e-mail vérifiée et l'authentification à deux facteurs, sans lesquelles Test PyPI refuse toute publication. Créez ce jeton dans les réglages de votre compte.

Le nom `thermometre` est sans doute déjà pris sur Test PyPI : dans `pyproject.toml`, que vous modifiez dans l'éditeur de fichiers (le panneau Fichiers de Colab, l'explorateur du Codespace), renommez le projet `thermometre-<votre pseudo>`, sans renommer le dossier `src/thermometre`, que vous indiquez alors à `uv_build`. Déclarez-y aussi Test PyPI comme index de publication, puis reconstruisez le paquet dans un `dist/` vide et publiez-le avec `uv publish`. Dans le notebook, ne tapez pas le jeton en clair : demandez-le avec `getpass`.

**Documentation utile**

- https://docs.astral.sh/uv/concepts/build-backend/#modules
- https://docs.astral.sh/uv/guides/package/#publishing-your-package
- https://docs.astral.sh/uv/concepts/indexes/#pinning-a-package-to-an-index
- https://docs.astral.sh/uv/reference/cli/#uv-publish--index
- https://docs.python.org/fr/3/library/getpass.html#getpass.getpass

In [ ]:
# Votre code ici

### Conseils supplémentaires

La table `[tool.uv.build-backend]` a une clé `module-name`, le nom du module à mettre dans la roue : `uv_build` cherche sinon `src/thermometre_<votre_pseudo>/`. Un index se déclare dans une table `[[tool.uv.index]]` : un nom (`testpypi`), l'`url` de lecture (`https://test.pypi.org/simple/`), la `publish-url` (`https://test.pypi.org/legacy/`) et `explicit = true`, pour que uv n'y cherche jamais les dépendances du projet. `uv publish --index testpypi --token {jeton}` publie le contenu de `dist/` sur cet index (IPython remplace `{jeton}` par la valeur de la variable). La cellule de la solution ne publie que si vous passez `PUBLIER` à `True`.

```toml
[tool.uv.build-backend]
module-name = ...

[[tool.uv.index]]
name = "testpypi"
url = ...
publish-url = ...
explicit = true
```

```python
!rm -rf dist
!uv build
jeton = getpass(...)
!uv publish ...
```

### Solution

Le `pyproject.toml`, avec votre pseudo :

```toml
[project]
name = "thermometre-monpseudo"
version = "0.2.0"
description = "Conversions de températures."
readme = "README.md"
requires-python = ">=3.12"
dependencies = []

[project.scripts]
vers-fahrenheit = "thermometre.cli:main"

[build-system]
requires = ["uv_build>=0.13.1,<0.14.0"]
build-backend = "uv_build"

[tool.uv.build-backend]
module-name = "thermometre"

[[tool.uv.index]]
name = "testpypi"
url = "https://test.pypi.org/simple/"
publish-url = "https://test.pypi.org/legacy/"
explicit = true
```

In [ ]:
from getpass import getpass

PUBLIER = False  # À passer à True pour publier sur Test PyPI

if PUBLIER:
  !rm -rf dist
  !uv build
  jeton = getpass("Jeton d'API Test PyPI : ")
  !uv publish --index testpypi --token {jeton}

Le paquet s'installe ensuite d'un autre environnement avec `uv pip install --index https://test.pypi.org/simple/ thermometre-<votre pseudo>`. Pour PyPI lui-même, la publication se fait d'ordinaire depuis la CI, sans jeton, par la publication de confiance (*Trusted Publishing*) : https://docs.pypi.org/trusted-publishers/.